|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The roofline<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: prefill and decode are two different machines<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root, wherever this notebook was opened from
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

The last notebook put a matmul on the roofline. Now put a real model on it.

One model, two jobs. Prefill reads a whole prompt at once. Decode produces
one token at a time. You will find that they live on opposite sides of the
ridge, and almost everything else in this course follows from that.

In [ ]:
### run this cell: the model, and what it weighs

model = AutoModelForCausalLM.from_pretrained(
          'Qwen/Qwen3-0.6B', dtype=torch.bfloat16).cuda().eval()

weight_bytes = sum(p.numel()*p.element_size() for p in model.parameters())
bandwidth    = cudalib.peak_bandwidth(fresh=True)

print(f'weights:   {weight_bytes/1e9:.2f} GB')
print(f'bandwidth: {bandwidth:.0f} GB/s')
print(f'so one read of the weights costs at least {1000*weight_bytes/1e9/bandwidth:.2f} ms')

# Exercise 1: how fast is prefill

Time a forward pass over a prompt of length L, with no cache, and report the
cost per token.

In [ ]:
@torch.inference_mode()
def prefill_ms(L):
  x = torch.randint(0, 1000, (1,L), device='cuda')
  return cudalib.bench_ms(lambda: model(x, use_cache=False),
                          iters=10, warmup=3, best_of=2)

print(f"{'prompt':>7} {'ms':>9} {'ms/token':>10}")
for L in [128,256,512,1024,2048]:
  ms = prefill_ms(L)
  print(f'{L:>7} {ms:>9.2f} {ms/L:>10.4f}')

# Exercise 2: how fast is decode

Now time a single extra token, with the context already in the cache. This is
what a server does for every token after the first.

In [ ]:
import copy

@torch.inference_mode()
def decode_ms(L):
  x    = torch.randint(0, 1000, (1,L), device='cuda')
  past = model(x, use_cache=True).past_key_values
  nxt  = torch.randint(0, 1000, (1,1), device='cuda')
  return cudalib.bench_ms(
      lambda: model(nxt, past_key_values=copy.copy(past), use_cache=True),
      iters=10, warmup=3, best_of=2)

print(f"{'context':>8} {'ms per token':>13}")
for L in [128,512,2048]:
  print(f'{L:>8} {decode_ms(L):>13.2f}')

# Exercise 3: against the floor

A decode step has to read every weight. That is a hard floor in milliseconds.
Work out the floor, compare it with what you measured, and turn the gap into
an achieved-bandwidth number.

In [ ]:
p_ms = prefill_ms(2048)/2048
d_ms = decode_ms(2048)

floor_ms = 1000 * weight_bytes/1e9 / bandwidth
achieved = (weight_bytes/1e9) / (d_ms/1000)

print(f'prefill: {p_ms:8.4f} ms/token')
print(f'decode:  {d_ms:8.4f} ms/token   ({d_ms/p_ms:.0f}x more, for the same model)')
print(f'\nthe floor for a decode step is {floor_ms:.2f} ms (one read of the weights)')
print(f'you measured                   {d_ms:.2f} ms')
print(f'achieved bandwidth             {achieved:.0f} GB/s = {100*achieved/bandwidth:.0f}% of peak')

### Three numbers, and what each one is telling you

**Decode costs hundreds of times more per token than prefill.** Same
weights, same arithmetic per weight. The only difference is how many
tokens share one read of the model. Prefill shares it across the whole
prompt; decode has one token to spread it over. They are not two speeds
of the same machine, they are two machines.

**Decode does not reach its own floor.** One read of the weights should
cost a few milliseconds and you measured several times that, so achieved
bandwidth comes out well under peak. On a 0.6B model the GPU finishes
each layer before Python has finished asking for the next one, and you
are timing the asking.

That gap is not a fact about the hardware. It is stage 12, and you close
it with CUDA graphs.

**The floor itself is real.** Batch 1 cannot beat it, whatever you do to
the code. The only way past it is to make one read of the weights serve
more than one token, which is stages 04 and 05.